<a href="https://colab.research.google.com/github/Sakinat-Folorunso/OOU_CSC309_Artificial_Intelligence/blob/main/notebooks/CSC309_Week10_AI_Languages_Vision_Robots_Student_Centred.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CSC 309 – Artificial Intelligence
**Week 10 Practical (PH):** AI programming languages, image processing, face verification and simple robots

**Instructor:** Dr Sakinat Folorunso · Associate Professor of AI Systems · Department of Computer Sciences, Olabisi Onabanjo University (OOU), Ago-Iwoye, Ogun State, Nigeria  
**Course:** CSC 309 (2 units; LH 15, PH 45) · **Mode:** student-centred, hands-on, Google Colab  
**CCMAS learning outcomes covered:** LO8

> 🎓 **Instructor note** – this notebook is the *STUDENT (TODO cells are stubs)* version.


## How to use this notebook
1. Fill in the **Group Log**, then do the **Do Now** with your partner.
2. Run the **Setup** cell once (*Runtime → Run all* is safe: nothing here crashes if a TODO is still empty).
3. Work top to bottom. Only edit cells marked **`# TODO(Student)`**.
4. Each task ends with an **auto-check** (✅/❌). Keep going until it is ✅.
5. Finish with the **Reflection**. Fast finishers: try the **Extensions**.

6. **Read the comments.** Every line of code has a plain-English comment above it. Read the comment first, predict what the line does, then run the cell and check your prediction.
7. Hand in the **Assignment** (marks and deadline are in the notebook) after the lab.

*Every random draw uses `SEED = 42`, so your numbers should match your classmates' numbers.*


## Lab tasks at a glance

| Part | Lab task(s) you must complete |
|---|---|
| Part A — Language cost: the same computation three ways | *Guided lab: run every cell, read the comments, explain the output in your own words* |
| Part B — Image processing from scratch | *Guided lab: run every cell, read the comments, explain the output in your own words* |
| Part C — Face verification (simulation) with FAR / FRR / EER | Task C2 — Is the system fair? Error rates *per group* (and consent) |
| Part D — Eigenfaces (PCA by SVD) | *Guided lab: run every cell, read the comments, explain the output in your own words* |
| Part E — Simple robots: reactive wall-following vs planning (A\*) | *Guided lab: run every cell, read the comments, explain the output in your own words* |
| Part F — Lawn-mower robot: coverage path planning (group-project core) | *Guided lab: run every cell, read the comments, explain the output in your own words* |
| **Assignment** | see **📝 Assignment / deliverables** near the end |
| **Reflection** | short written reflection at the end (5–7 sentences) |
| *Extensions* | optional, for fast finishers |

In [ ]:
#@title 🧑🏽‍🤝‍🧑🏾 Group Log (fill before you start)
# Form field: type your group members' names here
group_members = "Type names here"   #@param {type:"string"}
# Form field: assign roles (driver, navigator, recorder, presenter)
roles_notes   = "Driver / Navigator / Recorder / Presenter"  #@param {type:"string"}
# Print them so your notebook records who did what
print("👥 Group:", group_members); print("📝 Roles:", roles_notes)


In [ ]:
#@title 🔧 Setup (run once) — seed, logging, helpers
# Standard-library modules used by every lab (os, re, math, random, logging, time ...)
import sys, os, re, math, random, logging, subprocess, warnings, time, itertools
# NumPy: fast arrays and random numbers
import numpy as np
# One seed for the whole course so every run gives the same numbers
SEED = 42                                    # one seed for the whole course → reproducible results
# Seed both Python's and NumPy's random generators
random.seed(SEED); np.random.seed(SEED)
# Configure logging so messages print as "LEVEL | message"
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s", force=True)
# A named logger for this course
log = logging.getLogger("csc309")
# Hide noisy library warnings so output stays readable
warnings.filterwarnings("ignore")

def ensure(pip_name, import_name=None):
    """Import a package; pip-install it first if it is missing (Colab-friendly)."""
    # Try to import the library
    try:
        # Return the imported module
        return __import__(import_name or pip_name)
    # If it is not installed ...
    except ImportError:
        # ... tell the student we are installing it
        log.info("installing %s ...", pip_name)
        # ... and install it with pip
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])
        # Return the imported module
        return __import__(import_name or pip_name)

# List that stores every check result
_CHECKS = []
def check(name, condition, hint=""):
    """Auto-check. Never raises, so 'Run all' always finishes. Pass a lambda to catch your own bugs."""
    # Try to import the library
    try:
        # Run the check: call it if it is a function, otherwise use the value
        ok = bool(condition() if callable(condition) else condition)
        # Default: no error message
        err = ""
    # If the student's code crashes, we report instead of stopping the notebook
    except Exception as e:                    # student code may crash; report instead of stopping
        # Mark as failed and keep the error message
        ok, err = False, f"{type(e).__name__}: {e}"
    # Store the result
    _CHECKS.append((name, ok))
    # Print a ✅ or ❌ line with a hint on failure
    print(("✅ PASS       " if ok else "❌ TRY AGAIN  ") + name + ("" if ok else f"   → {err or hint}"))
    # Return whether the check passed
    return ok

# Print how many checks passed
def checks_summary():
    # Count the passed checks
    passed = sum(ok for _, ok in _CHECKS)
    # Print the total
    print(f"\nChecks passed: {passed}/{len(_CHECKS)}")

def mean_ci(x, conf=0.95):
    """Mean, sample std and a two-sided t-based confidence interval."""
    # SciPy for the t-distribution
    from scipy import stats
    # Convert to a float array and count the observations
    x = np.asarray(x, dtype=float); n = len(x)
    # Mean and sample standard deviation (n-1 divisor)
    m = x.mean(); s = x.std(ddof=1) if n > 1 else 0.0
    # Half-width of the 95% confidence interval from the t-distribution
    h = stats.t.ppf((1 + conf) / 2, n - 1) * s / math.sqrt(n) if n > 1 else 0.0
    # Return mean, SD, lower and upper confidence limits
    return m, s, m - h, m + h

# Print a one-line statistical summary
def report(label, x):
    # Compute the statistics
    m, s, lo, hi = mean_ci(x)
    # Print mean, SD, CI and n
    print(f"{label:<28} mean={m:8.2f}  sd={s:7.2f}  95% CI=[{lo:.2f}, {hi:.2f}]  (n={len(x)})")

def paired_wilcoxon(a, b):
    """Two-sided Wilcoxon signed-rank test for paired samples (same seeds/instances)."""
    # SciPy for the t-distribution
    from scipy import stats
    # Convert both samples to arrays
    a, b = np.asarray(a, float), np.asarray(b, float)
    # If the two samples are identical the test is undefined
    if np.allclose(a, b):
        # Report p = 1 (no difference)
        return 1.0
    # Paired Wilcoxon signed-rank test p-value
    return float(stats.wilcoxon(a, b).pvalue)

# Setup finished
print("✅ Setup complete — seed =", SEED)


### Learning objectives (with Bloom level)
| # | By the end of this lab you can… | Bloom | CCMAS LO |
|---|---|---|---|
| 1 | compare languages for AI (Python, Prolog, Lisp, C++/Julia) and **measure** the cost of an interpreted loop | Understand / Analyse | LO8 |
| 2 | implement 2-D **convolution** and edge detection from scratch | Apply | LO8 |
| 3 | evaluate a **face-verification** system with **FAR, FRR, EER** | Evaluate | LO8 |
| 4 | build **eigenfaces** (PCA by SVD) and a nearest-neighbour recogniser | Apply / Analyse | LO8 |
| 5 | compare a reactive robot (wall following) with a planner (A\*), and design a **lawn-mower coverage** path | Analyse / Create | LO8 |

**Why these languages?** *Prolog* — logic programming, unification and backtracking built in (Weeks 5, 9). *Lisp* — code as data, symbolic AI heritage. *Python* — ecosystem (NumPy, scikit-learn, PyTorch). *C++/Julia* — speed. The point of a language choice is **expressiveness × performance × ecosystem**, not fashion.

**Face verification metrics.** Given a similarity distance $d(x,y)$ and a threshold $\tau$ (accept iff $d\le\tau$):
$$\mathrm{FAR}(\tau)=\Pr[d\le\tau\mid\text{impostor pair}],\qquad \mathrm{FRR}(\tau)=\Pr[d>\tau\mid\text{genuine pair}].$$
Raising $\tau$ lowers FRR but raises FAR. The **equal error rate (EER)** is the common value at the crossing point.

**Do Now (3 min).** A bank's face login and a classroom attendance app: which error (FAR or FRR) is worse for each? Which threshold would you choose?

In [ ]:
#@title 📦 Imports and a seeded generator
# Plotting library
import matplotlib.pyplot as plt
# SciPy statistics
from scipy import stats
# Seeded random generator (reproducible)
rng = np.random.default_rng(SEED)

## Part A — Language cost: the same computation three ways
We compute $\sum_i x_i^2$ for $n=10^5$ numbers with (i) an explicit Python loop, (ii) a generator expression, (iii) NumPy. We time 15 repeats and report mean ± sd.

In [ ]:
#@title ⏱️ Timing experiment
# Timer library
import time
# 100,000 random numbers, as an array and as a Python list
x = rng.standard_normal(100_000); xl = x.tolist()
# Sum of squares with a plain Python loop
def loop_sum(v):
    # Running total
    s = 0.0
    # Add each square one at a time (slow: interpreted)
    for a in v: s += a * a
    # Return the total
    return s
# Same sum with a generator expression
def gen_sum(v):   return sum(a * a for a in v)
# Same sum with NumPy's dot product (vectorised, fast)
def np_sum(v):    return float(np.dot(v, v))
# Time a function over repeated runs
def timeit(fn, arg, reps=15):
    # List of timings
    ts = []
    # Repeat the measurement
    for _ in range(reps):
        # Start the clock, run, stop the clock
        t0 = time.perf_counter(); fn(arg); ts.append(time.perf_counter() - t0)
    # Return the timings in milliseconds
    return np.array(ts) * 1e3
# Time all three versions
T = {"python loop": timeit(loop_sum, xl), "generator": timeit(gen_sum, xl), "numpy": timeit(np_sum, x)}
# Print mean and standard deviation of each
for k, v in T.items(): print(f"{k:<12} {v.mean():8.3f} ± {v.std(ddof=1):6.3f} ms")
# Speed-up of NumPy over the loop
print(f"speed-up numpy vs loop ≈ {T['python loop'].mean() / T['numpy'].mean():.0f}×")
check("all three give the same value", lambda: abs(loop_sum(xl) - np_sum(x)) < 1e-6 and abs(gen_sum(xl) - np_sum(x)) < 1e-6)
check("NumPy is at least 10× faster than the explicit loop", lambda: T["python loop"].mean() / T["numpy"].mean() > 10)

**Discuss.** Interpreted loops are slow because of per-iteration dynamic dispatch; NumPy moves the loop into compiled code (*vectorisation*). In Prolog the same task is a recursive predicate `sum_sq([],0). sum_sq([X|T],S) :- sum_sq(T,S1), S is S1+X*X.` — elegant but not fast. *Choose the language for the part of the problem that dominates.*

## Part B — Image processing from scratch
A grey image is a matrix $I\in\mathbb{R}^{H\times W}$. A **valid 2-D cross-correlation** with kernel $K\in\mathbb{R}^{k\times k}$ is
$$(I\star K)[i,j]=\sum_{u=0}^{k-1}\sum_{v=0}^{k-1} I[i+u,\,j+v]\;K[u,v],\qquad \text{output size }(H-k+1)\times(W-k+1).$$
**Task.** Implement `conv2d`. Then detect edges with the Sobel kernels $G_x,G_y$ and magnitude $\sqrt{G_x^2+G_y^2}$.

### 🔍 Code walkthrough — convolution as shifted copies
Instead of looping over every pixel, `conv2d` loops only over the **kernel's 9 cells**. For each kernel weight it adds a *shifted copy of the whole image* times that weight. This is the same arithmetic as sliding the kernel, but the heavy work is done by NumPy on whole arrays, which is why it is fast. Check the output size: a $k\times k$ kernel shrinks each side by $k-1$.

In [ ]:
#@title 🖼️ conv2d (TODO) and a synthetic test image
def conv2d(img, K):
    """TODO(Student): valid cross-correlation of 2-D array `img` with kernel `K` (no padding, stride 1)."""
    # TODO: height, width and kernel size of the image
    H, W = img.shape; k = K.shape[0]
    # Placeholder output array of the right size
    return np.zeros((H - k + 1, W - k + 1))

# A toy image: a bright rectangle with a dim centre
img = np.zeros((40, 40)); img[10:30, 12:28] = 1.0; img[18:22, 18:22] = 0.4        # a bright rectangle with a dim centre
# Sobel kernels for horizontal and vertical edges
SX = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], float); SY = SX.T
# Edge strength = length of the gradient (gx, gy)
def sobel(im): return np.hypot(conv2d(im, SX), conv2d(im, SY))

In [ ]:
#@title ✅ Auto-check and picture
# SciPy reference implementation to check against
from scipy.signal import correlate2d
# Edge map of the image
E = sobel(img)
# Show the image and its edges side by side
fig, ax = plt.subplots(1, 2, figsize=(7, 3)); ax[0].imshow(img, cmap="gray"); ax[0].set_title("image"); ax[1].imshow(E, cmap="gray"); ax[1].set_title("Sobel magnitude")
# Hide the axes
for a in ax: a.axis("off")
# Display
plt.show()
# Random test image and kernel
rim = rng.random((9, 9)); K3 = rng.standard_normal((3, 3))
check("conv2d matches scipy.signal.correlate2d (valid)", lambda: np.allclose(conv2d(rim, K3), correlate2d(rim, K3, mode="valid")))
check("output shape is (H−k+1)×(W−k+1)",               lambda: conv2d(img, SX).shape == (38, 38))
check("edges strong on the rectangle border, ~0 inside the flat bright region", lambda: E[8:10, 14:26].max() > 2 and E[12:16, 14:18].max() < 1e-9)

## Part C — Face verification (simulation) with FAR / FRR / EER
We need face images. We try the **Olivetti faces** (40 people × 10 images, 64×64) via scikit-learn; if there is no internet we fall back to scikit-learn's bundled **digits** (10 "identities", 8×8) so that every metric below still runs. *The code is identical either way; only absolute numbers differ.* **Ethics:** use only datasets whose licence permits it, and never scrape classmates' photos without consent.

In [ ]:
#@title 🧑 Load faces (with offline fallback)
# Face and digit data sets from scikit-learn
from sklearn.datasets import fetch_olivetti_faces, load_digits
# Try the real faces first
try:
    # Load Olivetti faces
    d = fetch_olivetti_faces(random_state=42, shuffle=False); X, y, DSRC = d.data, d.target, "Olivetti faces (40 people × 10 images)"
# If it is unavailable ...
except Exception as e:
    # ... log a warning
    log.warning("Olivetti unavailable (%s): using digits as stand-in identities", type(e).__name__)
    # ... use the digits data instead
    d = load_digits(); X, y, DSRC = d.data / 16.0, d.target, "FALLBACK: sklearn digits (10 identities)"
# Print the data source and shapes
print(DSRC, "| X:", X.shape, "| identities:", len(np.unique(y)))
# stratified split: half the images of each identity for training, half for testing
# Train and test index lists
tr, te = [], []
# For each person
for c in np.unique(y):
    # Shuffle their images; half for training, the rest (max 40) for testing
    idx = np.where(y == c)[0]; idx = rng.permutation(idx); h = len(idx) // 2; tr += list(idx[:h]); te += list(idx[h:h + min(len(idx) - h, 40)])
# Convert to arrays
tr, te = np.array(tr), np.array(te)
# Print the split sizes
print("train", len(tr), "test", len(te))

### 🔍 Code walkthrough — the two ways a verifier fails
- **FAR** (false accept rate): an *impostor* pair is accepted because its distance is below the threshold τ.
- **FRR** (false reject rate): a *genuine* pair is rejected because its distance is above τ.

Raising τ lowers FRR but raises FAR. The **equal error rate** is the point where the two curves cross; `eer` finds it by trying every observed distance as a threshold.

In [ ]:
#@title 📏 FAR, FRR, EER (TODO)
def far_frr(genuine, impostor, tau):
    """TODO(Student): genuine and impostor are arrays of distances. Accept iff d <= tau.
       FAR = fraction of IMPOSTOR distances accepted; FRR = fraction of GENUINE distances rejected."""
    # Placeholder: FAR and FRR (replace)
    return 0.0, 0.0

def eer(genuine, impostor):
    """TODO(Student): scan candidate thresholds (all observed distances) and return (eer, tau) where |FAR−FRR| is smallest;
       eer = (FAR+FRR)/2 at that threshold."""
    # Placeholder values
    return 1.0, 0.0

In [ ]:
#@title ✅ Auto-check, then the pixel-space verifier
# Tiny test case
g, im = np.array([0.1, 0.2, 0.3, 0.4]), np.array([0.35, 0.5, 0.6, 0.7])
check("far_frr at τ=0.35: FAR=0.25 (0.35 accepted), FRR=0.25 (0.4 rejected)", lambda: far_frr(g, im, 0.35) == (0.25, 0.25))
check("τ=0 rejects everyone: FAR=0, FRR=1",      lambda: far_frr(g, im, 0.0) == (0.0, 1.0))
check("τ=∞ accepts everyone: FAR=1, FRR=0",      lambda: far_frr(g, im, 9.9) == (1.0, 0.0))
check("eer on separable data is 0",              lambda: abs(eer(g, np.array([0.8, 0.9, 1.0]))[0]) < 1e-9)

# Distance functions
from scipy.spatial.distance import pdist, squareform
# All pairwise distances among test images, and whether each pair is the same person
Xte, yte = X[te], y[te]; D = squareform(pdist(Xte)); iu = np.triu_indices(len(te), 1); same = (yte[:, None] == yte[None, :])[iu]
# Genuine and impostor distance lists
gen, imp = D[iu][same], D[iu][~same]
# Compute and print the EER
e, t = eer(gen, imp); print(f"pixel-space verifier: {len(gen)} genuine, {len(imp)} impostor pairs; EER = {e:.3f} at τ = {t:.2f}")
# FAR and FRR curves over many thresholds
taus = np.linspace(min(gen.min(), imp.min()), max(gen.max(), imp.max()), 200); curve = np.array([far_frr(gen, imp, q) for q in taus])
# Plot the two curves
plt.figure(figsize=(5, 3.2)); plt.plot(taus, curve[:, 0], label="FAR"); plt.plot(taus, curve[:, 1], label="FRR"); plt.axvline(t, ls=":", c="k")
# Labels and display
plt.xlabel("threshold τ"); plt.ylabel("error rate"); plt.legend(); plt.title(f"EER ≈ {e:.2f}"); plt.show()
check("verifier is better than chance (EER < 0.5)", lambda: e < 0.5)

### Task C2 — Is the system fair? Error rates *per group* (and consent)
A single EER hides who bears the errors. Suppose the same threshold is used for two groups of users, and that images of **group B** were captured in poorer lighting, so their genuine distances are larger and their impostor distances smaller. Using the pooled threshold $\tau$ found above, compute $\mathrm{FAR}$ and $\mathrm{FRR}$ for each group and the disparities $\mathrm{FAR}_B-\mathrm{FAR}_A$ and $\mathrm{FRR}_B-\mathrm{FRR}_A$. (This is a **simulation** of a mechanism, not a measurement of real people.)

**Ethics checklist (answer in writing before any real deployment).**
1. **Consent:** did each person agree freely and knowingly, and can they refuse without penalty? (The Nigeria Data Protection Act 2023 treats biometric data as sensitive.)
2. **Privacy and storage:** are photographs or only feature vectors stored, for how long, and who can read them?
3. **Fairness:** are FAR and FRR reported *per group* (skin tone, sex, age, lighting) and is there a stated limit on the gap?
4. **Misuse:** could the database be used to track people?
5. **Alternatives:** is a card, PIN or manual register good enough?

In [ ]:
# TODO(Student): per-group FAR/FRR at the pooled threshold t (use far_frr from above)
# Seeded generator
rg = np.random.default_rng(42)
# Randomly assign pairs to group A or B
half = rg.random(len(gen)) < 0.5; halfi = rg.random(len(imp)) < 0.5            # random group membership of each pair
# Group B has more variation between genuine pairs
genA, genB = gen[half], gen[~half] * 1.25                                      # group B: more variation between images of the same person
# ...and impostors look a little closer
impA, impB = imp[halfi], imp[~halfi] * 0.90                                    # ... and impostors look a little closer
# TODO: compute error rates for both groups
farA = frrA = farB = frrB = None                                               # compute with far_frr(genuine, impostor, t)

In [ ]:
#@title ✅ Auto-check for Task C2
# Only print if you have computed them
if farA is not None:
    # Print each group's FAR and FRR
    print(f"group A: FAR={farA:.3f} FRR={frrA:.3f}   group B: FAR={farB:.3f} FRR={frrB:.3f}")
    # Print the disparity between groups
    print(f"disparity  FAR_B-FAR_A = {farB-farA:+.3f}   FRR_B-FRR_A = {frrB-frrA:+.3f}")
check("rates are probabilities", lambda: all(0 <= v <= 1 for v in (farA, frrA, farB, frrB)))
check("group B suffers more false rejects (FRR_B > FRR_A)", lambda: frrB > frrA)
check("group B also suffers more false accepts (FAR_B > FAR_A)", lambda: farB > farA)

## Part D — Eigenfaces (PCA by SVD)
Centre the training matrix $X_c=X-\mathbf 1\mu^\top$ and take the SVD $X_c=U\Sigma V^\top$. The columns of $V$ are the **eigenfaces**; project $z=V_k^\top (x-\mu)$ onto the first $k$ components and classify with 1-NN in the $k$-dimensional space. The fraction of variance kept is $\sum_{i\le k}\sigma_i^2/\sum_i\sigma_i^2$.

In [ ]:
#@title 🧬 PCA via SVD (TODO) and 1-NN recogniser
def fit_pca(Xtr, k):
    """TODO(Student): return (mu, V_k) where mu is the mean row and V_k has shape (n_features, k) from the SVD of the centred data."""
    # Placeholder: mean and identity projection
    return Xtr.mean(0), np.eye(Xtr.shape[1])[:, :k]

# Project images onto k principal components
def project(Xm, mu, Vk): return (Xm - mu) @ Vk

# Nearest-neighbour classifier in the reduced space
def nn_predict(Ztr, ytr, Zte):
    # Squared distances, then the closest training example's label
    D2 = ((Zte[:, None, :] - Ztr[None, :, :]) ** 2).sum(-1); return ytr[D2.argmin(1)]

In [ ]:
#@title ✅ Auto-check and accuracy vs number of components
# scikit-learn PCA to compare with
from sklearn.decomposition import PCA
# Compare your PCA with the library's
mu, Vk = fit_pca(X[tr], 10); pca = PCA(10, svd_solver="full").fit(X[tr])
check("components are orthonormal (VᵀV = I)", lambda: np.allclose(Vk.T @ Vk, np.eye(10), atol=1e-6))
check("subspace agrees with sklearn PCA (|cos| ≈ 1 for each of the first 3 axes)", lambda: all(abs(Vk[:, i] @ pca.components_[i]) > 0.999 for i in range(3)))
# Results table
rows = []
# Try different numbers of components
for k in [2, 5, 10, 20, 40]:
    # Fit PCA on the training set only
    k = min(k, X.shape[1], len(tr) - 1); mu, Vk = fit_pca(X[tr], k)
    # Accuracy of nearest-neighbour in PCA space
    acc = np.mean(nn_predict(project(X[tr], mu, Vk), y[tr], project(X[te], mu, Vk)) == y[te]); n = len(te)
    # 95% Wilson CI for the accuracy
    ci = stats.binomtest(int(round(acc * n)), n).proportion_ci(0.95, method="wilson"); rows.append((k, acc, ci.low, ci.high))
    # Print the result
    print(f"k = {k:>2}: accuracy = {acc:.3f} (95% Wilson CI [{ci.low:.3f}, {ci.high:.3f}], n = {n})")
# Baseline: raw pixels
raw = np.mean(nn_predict(X[tr], y[tr], X[te]) == y[te]); print(f"raw pixels 1-NN: {raw:.3f}")
check("PCA with enough components is within 0.05 of raw-pixel accuracy", lambda: rows[-1][1] >= raw - 0.05)
check("accuracy with 2 components is lower than with the largest k", lambda: rows[0][1] < rows[-1][1])

**Detection vs recognition.** *Detection* answers "is there a face, and where?" (e.g. OpenCV's Haar cascades; modern CNN detectors); *verification* answers "same person?" (1-to-1); *identification* answers "who is this?" (1-to-N). Face systems show **demographic performance gaps** — always report FAR/FRR *per group* (cf. Week 8's Pidgin gap) and obtain consent.

## Part E — Simple robots: reactive wall-following vs planning (A\*)
A robot in a grid maze must reach the goal. **Wall following** (right-hand rule) uses only local sensing and no map; **A\*** uses the whole map. Compare path lengths over 30 random mazes (paired Wilcoxon).

In [ ]:
#@title 🤖 Maze generator, right-hand wall follower, and A*
# Heap for A*
import heapq
# Create a random maze that is guaranteed solvable
def make_maze(n, p, rs):
    # Repeat until a path exists
    while True:
        # Random walls; start and goal open
        g = (rs.random((n, n)) < p).astype(int); g[0, 0] = g[-1, -1] = 0
        # Accept if A* finds a path
        if astar_len(g) is not None: return g

# Length of the shortest path with A*
def astar_len(g):
    # Size, goal, frontier, best costs
    n = len(g); goal = (n - 1, n - 1); pq = [(2 * (n - 1), 0, (0, 0))]; best = {(0, 0): 0}
    # Loop while the frontier is not empty
    while pq:
        # Pop the best node
        f, c, s = heapq.heappop(pq)
        # Goal reached: return the cost
        if s == goal: return c
        # Skip stale entries
        if c > best.get(s, 1e9): continue
        # Four directions
        for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            # Neighbour coordinates
            r, q = s[0] + dr, s[1] + dc
            # Move if inside, free, and cheaper than before
            if 0 <= r < n and 0 <= q < n and g[r, q] == 0 and c + 1 < best.get((r, q), 1e9):
                # Push with f = cost + Manhattan distance
                best[(r, q)] = c + 1; heapq.heappush(pq, (c + 1 + abs(n - 1 - r) + abs(n - 1 - q), c + 1, (r, q)))
    # No path
    return None

# Right-hand wall-following agent (a simple reflex agent)
def wall_follow_len(g, max_steps=4000):
    # Position, heading, and direction vectors
    n = len(g); pos = (0, 0); d = 0; dirs = [(0, 1), (1, 0), (0, -1), (-1, 0)]       # E, S, W, N ; right-hand rule
    # Cell is free if inside the grid and not a wall
    free = lambda r, c: 0 <= r < n and 0 <= c < n and g[r, c] == 0
    # Move for up to max_steps
    for step in range(max_steps):
        # Goal reached: return the number of steps
        if pos == (n - 1, n - 1): return step
        # Try right, straight, left, then back
        for turn in (1, 0, -1, 2):                                                   # try right, straight, left, back
            # New heading and the cell it points at
            nd = (d + turn) % 4; r, c = pos[0] + dirs[nd][0], pos[1] + dirs[nd][1]
            # Move there if free
            if free(r, c): pos, d = (r, c), nd; break
    # No path
    return None

# Seeded generator; list of results
rs = np.random.default_rng(42); L = []
# 30 random mazes
for _ in range(30):
    # Make a maze and measure both agents
    g = make_maze(12, 0.25, rs); L.append((astar_len(g), wall_follow_len(g)))
# Replace failures with the cap 4000
L = np.array([(a, w if w is not None else 4000) for a, w in L], float)
# Report A* path length
print(f"A* path length  : {L[:,0].mean():.1f} ± {L[:,0].std(ddof=1):.1f}")
# Report wall-following length
print(f"wall-following  : {L[:,1].mean():.1f} ± {L[:,1].std(ddof=1):.1f}  (cap 4000 steps)")
# One-sided paired Wilcoxon: wall-following longer than A*
w = stats.wilcoxon(L[:, 1], L[:, 0], alternative="greater"); print(f"paired Wilcoxon (wall > A*): W = {w.statistic:.0f}, p = {w.pvalue:.2e}")
check("A* is optimal: never longer than wall following", lambda: np.all(L[:, 0] <= L[:, 1]))
check("wall following is significantly longer (p < 0.05)", lambda: w.pvalue < 0.05)

## Part F — Lawn-mower robot: coverage path planning (group-project core)
**Problem.** Cover every free cell of a rectangular lawn with obstacles (flower beds). A **boustrophedon** ("as the ox ploughs") sweep covers rows alternately left→right and right→left; when an obstacle blocks the way the robot must detour. Metrics: **coverage** $=\frac{|\text{visited free cells}|}{|\text{free cells}|}$ and **redundancy** $=\frac{\text{steps}}{|\text{free cells}|}-1$ (0 is ideal).

**Task.** Implement `mow(grid)`: sweep the rows boustrophedon-style; for each row consider only the *segments* of consecutive free cells; between segments, travel by shortest path (BFS) through free cells. Return the list of visited cells in order (moves of one cell at a time).

In [ ]:
#@title 🌿 mow() (TODO) — returns a cell-by-cell path starting at (0,0)
# deque for BFS
from collections import deque
# Shortest path between two cells
def bfs_path(g, a, b):
    # Grid size, parent table, queue
    n, m = g.shape; prev = {a: None}; q = deque([a])
    # Loop until the queue is empty
    while q:
        # Take the next cell
        s = q.popleft()
        # Stop at the target
        if s == b: break
        # Four directions
        for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            # Neighbour cell
            t = (s[0] + dr, s[1] + dc)
            # Add unvisited free neighbours
            if 0 <= t[0] < n and 0 <= t[1] < m and g[t] == 0 and t not in prev: prev[t] = s; q.append(t)
    # Start rebuilding the path at the target
    p = []; s = b
    # Walk back to the start
    while s is not None: p.append(s); s = prev[s]
    # Return the path in forward order
    return p[::-1]

def mow(g):
    """TODO(Student): boustrophedon coverage. Hint: for each row r, build segments of free cells; visit segments left→right on even rows
    and right→left on odd rows; use bfs_path to move from the current cell to the start of the next segment. Return list of cells."""
    # Placeholder path (replace with your coverage planner)
    return [(0, 0)]

In [ ]:
#@title ✅ Auto-check and coverage statistics over 20 random lawns
# Metrics for a coverage path
def metrics(g, path):
    # Set of free cells
    free = {(r, c) for r in range(g.shape[0]) for c in range(g.shape[1]) if g[r, c] == 0}
    # Is every move legal (free cell, one step)?
    legal = all(g[p] == 0 for p in path) and all(abs(a[0]-b[0]) + abs(a[1]-b[1]) == 1 for a, b in zip(path, path[1:]))
    # Coverage, redundancy and legality
    return len(set(path) & free) / len(free), len(path) / len(free) - 1, legal
# Test on an empty room
empty = np.zeros((6, 8), int); cov, red, legal = metrics(empty, mow(empty))
check("empty lawn: full coverage, legal moves", lambda: cov == 1.0 and legal)
check("empty lawn: zero redundancy (each cell once)", lambda: red < 0.03)
# Test with an obstacle
obs = empty.copy(); obs[2:4, 3:5] = 1; cov2, red2, legal2 = metrics(obs, mow(obs))
check("lawn with a flower bed: still full coverage via detours", lambda: cov2 == 1.0 and legal2)
# Seeded generator; results list
rs = np.random.default_rng(42); R = []
# 20 random rooms
for _ in range(20):
    # Empty room
    g = np.zeros((10, 12), int); 
    # Place three random obstacles
    for _b in range(3): r0, c0 = rs.integers(1, 8), rs.integers(1, 10); g[r0:r0 + rs.integers(1, 3), c0:c0 + rs.integers(1, 3)] = 1
    # Keep the start free
    g[0, 0] = 0
    # Store the metrics
    R.append(metrics(g, mow(g)))
# Mean redundancy with 95% CI
R = np.array(R, float); mc, _sd, lo, hi = mean_ci(R[:, 1])
# Print the summary
print(f"coverage = {R[:,0].mean():.3f} (min {R[:,0].min():.3f}); redundancy = {mc:.3f} (95% CI [{lo:.3f}, {hi:.3f}]); all paths legal: {bool(R[:,2].all())}")
check("coverage is 1.0 on all connected random lawns", lambda: R[:, 0].min() > 0.99 and R[:, 2].all())

## Group project — *the lawn-mower robot* (CCMAS lab work)
**Teams of 4–5. Deliverable:** a 5-page report + the notebook + a 3-minute demo.

1. **Specification** (PEAS): performance, environment, actuators, sensors. State which of the Week 2 environment properties hold (partially observable? dynamic because of pets?).
2. **Algorithm:** extend `mow()` with (a) *unknown* obstacles discovered by a bump sensor (re-plan online), (b) a **battery** that forces a return to the charger (use A\*), (c) a second robot with **message passing** to divide the lawn (Week 11).
3. **Evaluation:** ≥ 30 random lawns (seed 42); report coverage, redundancy and energy as mean ± sd with a paired test between two strategies.
4. **Safety & ethics:** what if a child or animal enters the lawn? Describe a stop rule and test it.

*Rubric (20):* specification 4 · algorithm 6 · evaluation 6 · safety 2 · teamwork log 2.

## 📝 Assignment (5 marks, individual, formative)
1. *(1)* Apply a **new kernel** (Gaussian blur or Laplacian) with your `conv2d` and describe what changes in the image.
2. *(2)* Report FAR, FRR and EER for the face verifier, then give FAR and FRR **per group** with the disparity. State one consent or privacy safeguard you would require.
3. *(2)* Run `mow` on 20 random lawns with obstacles: coverage and redundancy as mean ± sd with 95% CI, and say whether every path was legal.

*Marked with the course rubric (class note, Table 0.3): correctness, evidence (numbers with CI and a named test), code clarity, interpretation. Submit the **executed notebook** (Runtime → Run all, outputs visible) as a Colab share link or `.ipynb` before the next lab.*

## Reflection
- Why is wall following *complete* only in some mazes, while A\* is complete on every finite grid?
- Which Part C error would matter most for a lecture-attendance app, and what threshold would you choose?

```text
Write your reflection here…
```

## Extensions
- Replace the pixel features with **HOG** or a pretrained CNN embedding and compare EER.
- Detect faces with OpenCV Haar cascades (`cv2.data.haarcascades`) on a photo you own — with consent.
- Add **diagonal** moves and non-uniform terrain cost to the coverage planner.

In [ ]:
checks_summary()  # how many auto-checks have you passed?